In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### Data Imports

In [31]:
# time series data
df_dur = pd.read_csv("../data/duration.csv")
df_ep = pd.read_csv("../data/e_price.csv")
df_occ = pd.read_csv("../data/occupancy.csv")
df_sp = pd.read_csv("../data/s_price.csv")
df_vol = pd.read_csv("../data/volume.csv")
# info + weather
inf  = pd.read_csv("../data/inf.csv")
w_airport = pd.read_csv("../data/weather_airport.csv")
w_central = pd.read_csv("../data/weather_central.csv")

In [34]:
w_airport.rename(columns={w_airport.columns[0]: 'time'}, inplace=True)
w_central.rename(columns={w_central.columns[0]: 'time'}, inplace=True)

### EDA

In [27]:
inf['station_id'] = inf['station_id'].astype(str)
cap = inf.set_index('station_id')['pile_count'] 

occ_rate = df_occ.copy()
for station in occ.columns:
    if station in cap.index:
        occ_rate[station] = occ[station] / cap[station]
occ_rate.head()

,time,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2673,2674,2675,2676,2677,2678,2679,2680,2681,2682
0,2022-09-01 00:00:00,0.20,0.409091,0.333333,0.909091,0.454545,0.4,0.0,0.454545,0.500000,...,0.444444,0.25,0.4,0.555556,0.500000,0.0,0.333333,0.166667,0.2,0.0
1,2022-09-01 01:00:00,0.35,0.272727,0.333333,0.545455,0.454545,0.7,0.0,0.431818,0.652778,...,0.694444,0.50,1.0,0.666667,0.583333,0.0,0.250000,0.250000,0.3,0.0
2,2022-09-01 02:00:00,0.35,0.136364,0.166667,0.909091,0.454545,0.8,0.0,0.454545,0.833333,...,0.472222,0.25,0.0,0.555556,0.666667,0.0,0.583333,0.250000,0.1,0.0
3,2022-09-01 03:00:00,0.40,0.136364,0.166667,0.909091,0.454545,0.8,0.0,0.454545,0.805556,...,0.472222,0.25,1.0,0.555556,0.666667,0.0,0.583333,0.250000,0.1,0.0
4,2022-09-01 04:00:00,0.30,0.272727,0.333333,0.727273,0.454545,0.7,0.0,0.431818,0.750000,...,0.694444,0.50,1.0,0.555556,0.583333,0.0,0.250000,0.250000,0.2,0.0


In [37]:
w_airport.head()

,time,T,P0,P,U,DD,Ff,ff10,WW,W'W',nRAIN,c,VV,Td
time,,,,,,,,,,,,,,
2023-08-31 23:00:00,31.08.2023 23:00,28.0,752.0,752.3,74,Wind blowing from the north,5,NaN,NaN,NaN,0,No Significant Clouds,10.0 and more,23.0
2023-08-31 22:00:00,31.08.2023 22:00,28.0,752.0,752.3,74,Wind blowing from the north,4,NaN,NaN,NaN,0,Few clouds (10-30%) 1500 m,10.0 and more,23.0
2023-08-31 21:00:00,31.08.2023 21:00,28.0,752.0,752.3,84,Wind blowing from the north-northwest,2,NaN,NaN,NaN,0,Scattered clouds (40-50%) 1500 m,10.0 and more,25.0
2023-08-31 20:00:00,31.08.2023 20:00,28.0,752.0,752.3,89,Wind blowing from the north-east,1,NaN,NaN,NaN,0,"Few clouds (10-30%) 450 m, broken clouds (60-9...",10.0 and more,26.0
2023-08-31 19:00:00,31.08.2023 19:00,27.0,752.0,752.3,94,Wind blowing from the east-northeast,5,NaN,"Heavy shower(s), rain",NaN,3,"Few clouds (10-30%) 450 m, few clouds (10-30%)...",4,26.0


In [36]:
w_airport.index = pd.to_datetime(w_airport['time'])  
w_central.index = pd.to_datetime(w_central['time'])

w_airport = w_airport.drop(['DD','WW']).resample('h').mean()
w_central = w_central.drop(['DD','WW']).resample('h').mean()

w_airport.columns = ['ap_' + c for c in w_airport.columns]
w_central.columns = ['ct_' + c for c in w_central.columns]

weather = pd.concat([w_airport, w_central], axis=1)
weather = weather.reindex(df_occ.index, method='nearest')  # align to occupancy index

C:\Users\anyak\AppData\Local\Temp\ipykernel_18532\3979654820.py:1: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  w_airport.index = pd.to_datetime(w_airport['time'])
C:\Users\anyak\AppData\Local\Temp\ipykernel_18532\3979654820.py:2: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  w_central.index = pd.to_datetime(w_central['time'])


KeyError: "['DD', 'WW'] not found in axis"

In [ ]:
data = {
    'occ':      df_occ,        
    'occ_rate': occ_rate,  
    'dur':      df_dur,
    'vol':      df_vol,
    'ep':       df_ep,
    'sp':       sp,
    'weather':  weather,
    'inf':      inf.set_index('station_id'),
}

# Convenience: shared timestamp index and station list
TIMESTAMPS = occ.index
STATIONS   = occ.columns.tolist()

print(f"Ready: {len(TIMESTAMPS)} hours × {len(STATIONS)} stations")

In [3]:
df_price = df_e_price + df_s_price

In [9]:
df = df_price.stack().reset_index(name='price')
df = df.merge(df_occupancy.stack().reset_index(name='occupancy'))
df = df.merge(df_volume.stack().reset_index(name='volume'))
df = df.merge(df_duration.stack().reset_index(name='duration'))

In [10]:
df.head()

,level_0,level_1,price,occupancy,volume,duration
0,0,time,2022-09-01 00:00:002022-09-01 00:00:00,2022-09-01 00:00:00,2022-09-01 00:00:00,2022-09-01 00:00:00
1,0,1001,1.67,4.0,4.083333,0.583333
2,0,1002,1.66,9.0,12.833333,1.833333
3,0,1003,1.86,2.0,2.333333,0.333333
4,0,1004,1.08,10.0,94.833333,4.083333


In [7]:
df[['price','occupancy','volume','duration']].corr()

ValueError: could not convert string to float: '2022-09-01 01:00:002022-09-01 01:00:00'

In [ ]:


df['log_price'] = np.log(df['price'] + 1e-6)
df['log_volume'] = np.log(df['volume'] + 1e-6)

X = sm.add_constant(df['log_price'])
model = sm.OLS(df['log_volume'], X).fit()
print(model.summary())

In [3]:
df_occupancy.head()

,time,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2673,2674,2675,2676,2677,2678,2679,2680,2681,2682
0,2022-09-01 00:00:00,4.0,9.0,2.0,10.0,5.0,4.0,0.0,20.0,36.0,...,16.0,1.0,4.0,5.0,6.0,0.0,4.0,2.0,2.0,0.0
1,2022-09-01 01:00:00,7.0,6.0,2.0,6.0,5.0,7.0,0.0,19.0,47.0,...,25.0,2.0,10.0,6.0,7.0,0.0,3.0,3.0,3.0,0.0
2,2022-09-01 02:00:00,7.0,3.0,1.0,10.0,5.0,8.0,0.0,20.0,60.0,...,17.0,1.0,0.0,5.0,8.0,0.0,7.0,3.0,1.0,0.0
3,2022-09-01 03:00:00,8.0,3.0,1.0,10.0,5.0,8.0,0.0,20.0,58.0,...,17.0,1.0,10.0,5.0,8.0,0.0,7.0,3.0,1.0,0.0
4,2022-09-01 04:00:00,6.0,6.0,2.0,8.0,5.0,7.0,0.0,19.0,54.0,...,25.0,2.0,10.0,5.0,7.0,0.0,3.0,3.0,2.0,0.0


In [4]:
df_e_price.head(20)

,time,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2673,2674,2675,2676,2677,2678,2679,2680,2681,2682
0,2022-09-01 00:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
1,2022-09-01 01:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
2,2022-09-01 02:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
3,2022-09-01 03:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
4,2022-09-01 04:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
5,2022-09-01 05:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
6,2022-09-01 06:00:00,1.2,1.0,1.1,0.32,0.79,0.2962,0.7402,1.0,0.3067,...,0.7,0.34,0.35,0.72,0.72,0.0,0.72,0.72,1.0,1.05
7,2022-09-01 07:00:00,1.2,1.0,1.1,0.32,0.79,0.7402,0.7402,1.0,0.2867,...,0.7,0.78,0.65,0.72,0.72,0.0,0.72,0.72,1.0,1.05
8,2022-09-01 08:00:00,1.2,1.0,1.1,0.79,0.79,0.7402,0.7402,1.0,0.7716,...,0.7,0.78,0.65,0.72,0.72,0.0,0.72,0.72,1.0,1.05
9,2022-09-01 09:00:00,1.2,1.0,1.1,0.79,0.79,1.0927,0.7402,1.0,0.7716,...,0.7,1.13,0.95,0.72,0.72,0.0,0.72,0.72,1.0,1.05


In [5]:
df_s_price.head()

,time,1001,1002,1003,1004,1005,1006,1007,1008,1009,...,2673,2674,2675,2676,2677,2678,2679,2680,2681,2682
0,2022-09-01 00:00:00,0.47,0.66,0.76,0.76,0.29,1.1438,1.1498,0.66,0.8733,...,0.76,0.82,0.73,0.76,0.76,0.52,0.76,0.76,0.52,0.7
1,2022-09-01 01:00:00,0.47,0.66,0.76,0.76,0.29,1.1438,1.1498,0.66,0.8733,...,0.76,0.82,0.73,0.76,0.76,0.52,0.76,0.76,0.52,0.7
2,2022-09-01 02:00:00,0.47,0.66,0.76,0.76,0.29,1.1438,1.1498,0.66,0.8733,...,0.76,0.82,0.73,0.76,0.76,0.52,0.76,0.76,0.52,0.7
3,2022-09-01 03:00:00,0.47,0.66,0.76,0.76,0.29,1.1438,1.1498,0.66,0.8733,...,0.76,0.82,0.73,0.76,0.76,0.52,0.76,0.76,0.52,0.7
4,2022-09-01 04:00:00,0.47,0.66,0.76,0.76,0.29,1.1438,1.1498,0.66,0.8733,...,0.76,0.82,0.73,0.76,0.76,0.52,0.76,0.76,0.52,0.7
